# 03 · STGCN on METR-LA

Spatio-Temporal Graph Convolutional Network (Yu, Yin & Zhu, IJCAI 2018), implemented in `src/models/stgcn/`.

Runs on **Google Colab** (use a GPU runtime) or **locally** (CUDA, Apple MPS or CPU).
Requires the processed files from `01_data_preprocess.ipynb`.

Current scope: build the graph and the model, run a forward pass, and run a short training
smoke test to confirm the model learns. Full training and evaluation come next.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mansiguptacs/CMPE_258_TrafficForecastingUsing_GraphNN.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/CMPE_258_TrafficForecastingUsing_GraphNN")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tables", "pyyaml"], check=True)
else:
    repo = Path.cwd()
    while not (repo / "configs" / "metr_la.yaml").exists():
        if repo.parent == repo:
            raise RuntimeError("Start this notebook from inside the project repository")
        repo = repo.parent

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Colab" if IN_COLAB else "Local", "| repo:", repo)

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import load_config, processed_dir
from src.data.pipeline import load_processed, make_datasets, PROCESSED_FILE
from src.models.stgcn.graph import get_adjacency, scaled_laplacian
from src.models.stgcn.model import STGCN
from src.models.losses import masked_mae

cfg = load_config()
if not (processed_dir(cfg) / PROCESSED_FILE).exists():
    raise FileNotFoundError("Processed data not found. Run 01_data_preprocess.ipynb first.")

torch.manual_seed(cfg["seed"])
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

data = load_processed(cfg)
datasets = make_datasets(cfg, data)
print("device:", device)
print({split: len(ds) for split, ds in datasets.items()}, "windows")

## 1. Graph

Chebyshev graph convolution needs the scaled Laplacian
$\tilde L = 2L/\lambda_{max} - I$, with $L = I - D^{-1/2} W D^{-1/2}$.
The adjacency is chosen by `graph.adjacency` in the config (`distance` or `correlation`).
The road adjacency is directed, so it is symmetrized for STGCN.

In [ ]:
adj = get_adjacency(cfg, data)
lap = scaled_laplacian(adj)
eig = np.linalg.eigvalsh(lap)

sym = np.maximum(adj, adj.T)
np.fill_diagonal(sym, 0)
degree = (sym > 0).sum(axis=1)
print(f"adjacency: {cfg['graph']['adjacency']}, {int((sym > 0).sum() // 2)} undirected edges")
print(f"neighbours per sensor: mean {degree.mean():.1f}, min {degree.min()}, max {degree.max()}")
print(f"scaled Laplacian eigenvalues in [{eig.min():.2f}, {eig.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(sym > 0, cmap="Greys", interpolation="nearest")
axes[0].set(title="Sensor connectivity (symmetrized)", xlabel="sensor", ylabel="sensor")
axes[1].hist(degree, bins=20)
axes[1].set(title="Neighbours per sensor", xlabel="degree", ylabel="sensors")
plt.tight_layout(); plt.show()

## 2. Model

Two ST-Conv blocks, each: gated temporal conv → Chebyshev graph conv → gated temporal conv → layer norm.
With 12 input steps and temporal kernel 3, the blocks shrink time from 12 to 4; the output layer
collapses the remaining 4 steps and predicts all 12 future steps at once.

In [ ]:
s, w = cfg["stgcn"], cfg["windows"]
model = STGCN(
    num_nodes=cfg["dataset"]["num_nodes"],
    lap=torch.from_numpy(lap),
    input_len=w["input_len"],
    output_len=w["output_len"],
    blocks=tuple(map(tuple, s["blocks"])),
    kt=s["kt"], ks=s["ks"], dropout=s["dropout"],
).to(device)
print(model)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Forward pass

The model predicts normalized speeds; they are converted back to mph with the per-sensor
training mean and std before comparing with targets. Missing targets are masked out.

In [ ]:
mean = torch.tensor(data["scaler"].mean, device=device)
std = torch.tensor(data["scaler"].std, device=device)

def to_mph(pred):
    return pred * std + mean

train_loader = DataLoader(datasets["train"], batch_size=s["batch_size"], shuffle=True)
x, y, y_mask = (t.to(device) for t in next(iter(train_loader)))
with torch.no_grad():
    pred = to_mph(model(x))
print("input :", tuple(x.shape), "[batch, time, sensors, features]")
print("output:", tuple(pred.shape), "[batch, future steps, sensors]")
print(f"untrained masked MAE: {masked_mae(pred, y, y_mask).item():.2f} mph")

## 4. Training smoke test

A few hundred optimizer steps to confirm gradients flow and the loss drops.
**These numbers are not results**; full training with early stopping on the validation split comes next.

In [ ]:
SMOKE_STEPS = 200
optimizer = torch.optim.Adam(model.parameters(), lr=s["lr"], weight_decay=s["weight_decay"])

model.train()
losses, start = [], time.time()
for step, (x, y, y_mask) in enumerate(train_loader):
    x, y, y_mask = x.to(device), y.to(device), y_mask.to(device)
    loss = masked_mae(to_mph(model(x)), y, y_mask)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step + 1 == SMOKE_STEPS:
        break
print(f"{SMOKE_STEPS} steps in {time.time() - start:.0f}s | "
      f"MAE first 10 steps {np.mean(losses[:10]):.2f} -> last 10 steps {np.mean(losses[-10:]):.2f} mph")

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.title("Smoke-test training loss"); plt.xlabel("step"); plt.ylabel("masked MAE (mph)")
plt.tight_layout(); plt.show()

In [ ]:
model.eval()
val_loader = DataLoader(datasets["val"], batch_size=s["batch_size"])
horizons = w["horizons"]
abs_err = {h: 0.0 for h in horizons}
count = {h: 0.0 for h in horizons}
with torch.no_grad():
    for x, y, y_mask in val_loader:
        x, y, y_mask = x.to(device), y.to(device), y_mask.to(device)
        pred = to_mph(model(x))
        for h in horizons:
            m = y_mask[:, h - 1].float()
            abs_err[h] += (torch.abs(pred[:, h - 1] - y[:, h - 1]) * m).sum().item()
            count[h] += m.sum().item()

for h in horizons:
    print(f"val MAE @ {h * cfg['dataset']['interval_minutes']:2d} min: {abs_err[h] / count[h]:.2f} mph  (after smoke test only)")

## Next

- Full training loop with validation-based early stopping and checkpointing.
- MAE / RMSE / masked MAPE at 15, 30 and 60 minutes on the test split, using the shared evaluation code.
- Switch `graph.adjacency` to `correlation` once the correlation graph is built, and compare.
- Missing-data robustness with `src.data.missing.apply_missing_mask`.